# EDA — synthetic music-context corpus

Artist-disjoint stand-in for FMA + MusicCaps + DEAM. Inspect tag rates, graph sizes, and a chroma heatmap.

In [ ]:
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
sys.path.insert(0, str(ROOT))
from src.taxonomy import GENRES, TAGS
from src.synthetic import load_cached_corpus

tracks = load_cached_corpus()
splits = json.loads((ROOT / 'data/splits/splits.json').read_text())
print({k: len(v) for k, v in splits.items()}, 'n=', len(tracks), 'feat', tracks[0]['seg_x'].shape)

In [ ]:
df = pd.DataFrame({
    'id': [t['id'] for t in tracks],
    'genre': [t['genre'] for t in tracks],
    'artist': [t['artist_id'] for t in tracks],
    'n_seg': [t['seg_x'].shape[0] for t in tracks],
    'n_chord': [t['chord_x'].shape[0] for t in tracks],
    'n_tags': [int(np.asarray(t['tags']).sum()) for t in tracks],
    'valence': [t['valence'] for t in tracks],
    'arousal': [t['arousal'] for t in tracks],
})
display(df.groupby('genre')[['n_seg','n_chord','n_tags','valence','arousal']].mean().round(3))

In [ ]:
tag_mat = np.stack([t['tags'] for t in tracks])
rates = tag_mat.mean(0)
order = np.argsort(-rates)
plt.figure(figsize=(10, 3.5))
plt.bar([TAGS[i] for i in order], rates[order], color='#c45c26')
plt.xticks(rotation=55, ha='right')
plt.ylabel('tag prevalence')
plt.tight_layout()
plt.show()

In [ ]:
t = tracks[0]
plt.figure(figsize=(6, 2.4))
plt.imshow(t['chroma_small'], aspect='auto', origin='lower', cmap='magma')
plt.title(f"{t['title']} · {t['genre']} · {t['caption'][:80]}…")
plt.ylabel('chroma')
plt.tight_layout()
plt.show()
print('chord path', t['chord_seq'])
print('tags', t['tag_names'])